In [1]:
import os
os.makedirs("src", exist_ok=True)

In [2]:
%%writefile src/train.py

import argparse
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import xgboost as xgb

# ---------- Training ----------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    args = parser.parse_args()

    input_file = [f for f in os.listdir(args.train) if f.endswith(".csv")][0]
    df = pd.read_csv(os.path.join(args.train, input_file))

    categorical_cols = [
        "gender", "MultipleLines", "InternetService", "OnlineSecurity",
        "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
        "StreamingMovies", "Contract", "PaymentMethod"
    ]
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

    X = df.drop("Churn", axis=1)
    y = df["Churn"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        eval_metric="logloss",
        random_state=42
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
    print(f"Precision: {precision_score(y_test, preds):.4f}")
    print(f"Recall: {recall_score(y_test, preds):.4f}")
    print(f"AUC: {roc_auc_score(y_test, probs):.4f}")

    joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))
    joblib.dump(list(X.columns), os.path.join(args.model_dir, "feature_columns.joblib"))

# ---------- Inference (used by the deployed endpoint) ----------
def model_fn(model_dir):
    model = joblib.load(os.path.join(model_dir, "model.joblib"))
    feature_columns = joblib.load(os.path.join(model_dir, "feature_columns.joblib"))
    return {"model": model, "feature_columns": feature_columns}

def input_fn(request_body, request_content_type):
    if request_content_type == "application/json":
        import json
        data = json.loads(request_body)
        return pd.DataFrame(data if isinstance(data, list) else [data])
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model_artifacts):
    model = model_artifacts["model"]
    feature_columns = model_artifacts["feature_columns"]

    # Align incoming data to the exact columns/order used at training time
    input_data = input_data.reindex(columns=feature_columns, fill_value=0)

    preds = model.predict(input_data)
    probs = model.predict_proba(input_data)[:, 1]
    return {"predictions": preds.tolist(), "probabilities": probs.tolist()}

def output_fn(prediction, response_content_type):
    import json
    return json.dumps(prediction)

if __name__ == "__main__":
    main()

Overwriting src/train.py


In [3]:
!cat src/train.py


import argparse
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import xgboost as xgb

# ---------- Training ----------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    args = parser.parse_args()

    input_file = [f for f in os.listdir(args.train) if f.endswith(".csv")][0]
    df = pd.read_csv(os.path.join(args.train, input_file))

    categorical_cols = [
        "gender", "MultipleLines", "InternetService", "OnlineSecurity",
        "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
        "StreamingMovies", "Contract", "PaymentMethod"
    ]
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

    X = df.drop("Churn", axis=1)
    y

In [4]:
%%writefile src/requirements.txt
xgboost==1.5.2

Overwriting src/requirements.txt


In [5]:
import sagemaker
from sagemaker.sklearn.estimator import SKLearn

session = sagemaker.Session()
role = "arn:aws:iam::276594885984:role/TelcoChurn-SageMaker-Role"

bucket = "telcom-churn"
processed_data_s3 = f"s3://{bucket}/processed/"
output_path = f"s3://{bucket}/model-artifacts/"

estimator = SKLearn(
    entry_point="train.py",
    role=role,
    instance_type="local",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    output_path=output_path,
    source_dir="src",
    dependencies=[]
)

estimator.fit({"train": processed_data_s3})

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


INFO:sagemaker:Creating training-job with name: sagemaker-scikit-learn-2026-07-15-08-06-08-924
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' is not installed. Proceeding to check for 'docker-compose' CLI.
INFO:sagemaker.local.image:'Docker Compose' found using Docker Compose CLI.
INFO:sagemaker.local.local_session:Starting training job
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
INFO:sagemaker.local.image:No AWS credentials found in session but credentials from EC2 Metadata Service are available.
INFO:sagemaker.local.

 Container ppibqoxwsi-algo-1-mup9s  Creating
 Container ppibqoxwsi-algo-1-mup9s  Created
Attaching to ppibqoxwsi-algo-1-mup9s
ppibqoxwsi-algo-1-mup9s  | /miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
ppibqoxwsi-algo-1-mup9s  |   import pkg_resources
ppibqoxwsi-algo-1-mup9s  | 2026-07-15 08:06:10,611 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
ppibqoxwsi-algo-1-mup9s  | 2026-07-15 08:06:10,619 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
ppibqoxwsi-algo-1-mup9s  | 2026-07-15 08:06:10,623 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
ppibqoxwsi-algo-1-mup9s  | 2026-07-15 08:06:10,633 sagemaker-trainin

INFO:sagemaker.local.image:===== Job Complete =====


ppibqoxwsi-algo-1-mup9s exited with code 0
Aborting on container exit...
 Container ppibqoxwsi-algo-1-mup9s  Stopping
 Container ppibqoxwsi-algo-1-mup9s  Stopped


In [6]:
from sagemaker.sklearn.model import SKLearnModel
import sagemaker

real_session = sagemaker.Session()  # forces real AWS, not local

sklearn_model = SKLearnModel(
    model_data=estimator.model_data,   # points to the S3 model.tar.gz from training
    role=role,
    entry_point="train.py",
    source_dir="src",
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=real_session
)

predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="telco-churn-endpoint"
)

INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2026-07-15-08-06-24-367
INFO:sagemaker:Creating endpoint-config with name telco-churn-endpoint
INFO:sagemaker:Creating endpoint with name telco-churn-endpoint


------!

In [7]:
print(estimator.model_data)

s3://telcom-churn/model-artifacts/sagemaker-scikit-learn-2026-07-15-08-06-08-924/output/model.tar.gz


In [8]:
import boto3
sm_client = boto3.client("sagemaker", region_name="ap-south-1")
response = sm_client.describe_endpoint(EndpointName="telco-churn-endpoint")
print(response["EndpointStatus"])

InService


In [9]:
import boto3
import json

runtime = boto3.client("sagemaker-runtime", region_name="ap-south-1")

sample = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": 1,
    "Dependents": 0,
    "tenure": 1,
    "PhoneService": 0,
    "MultipleLines": "No phone service",
    "InternetService": "DSL",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": 1,
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 29.85,
    "TotalCharges": 29.85
}

response = runtime.invoke_endpoint(
    EndpointName="telco-churn-endpoint",
    ContentType="application/json",
    Body=json.dumps(sample)
)

result = json.loads(response["Body"].read().decode())
print(result)

{'predictions': [0], 'probabilities': [0.4104595482349396]}
